# 22 - Reflection / Self-Correction

Reflection adds a checking stage after an agent produces an intermediate result.

Basic pattern:

```text
Generate
 ↓
Critic / verifier
 ↓
Good?
 ├─ yes → finish
 └─ no → revise
```

This can be implemented with a second model call, a deterministic checker, or a specialized verification tool.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU pydantic

## 1. Why reflection?

LLMs can:
- miss constraints
- make arithmetic mistakes
- overlook evidence
- format output incorrectly

A verifier can catch some of these problems before final output.

## 2. Deterministic reflection

Use deterministic checks whenever possible.

Example:
- JSON schema validation
- total equals sum of parts
- every citation points to a retrieved source
- email recipient is authorized

In [ ]:
def verify_total(items, stated_total):
    calculated = sum(items)
    return calculated == stated_total

items = [20, 30, 50]
print(verify_total(items, 100))
print(verify_total(items, 110))

## 3. Critic as structured output

A critic should produce a small structured judgment rather than a vague paragraph.

In [ ]:
from pydantic import BaseModel, Field

class Critique(BaseModel):
    pass_check: bool
    score: float = Field(ge=0, le=1)
    issue: str

example = Critique(
    pass_check=False,
    score=0.4,
    issue='The answer includes a claim not present in the evidence.',
)

print(example.model_dump())

## 4. Revision loop

```text
draft
 ↓
critic
 ↓
revision
 ↓
critic
 ↓
final
```

Always cap the number of reflection rounds.

In [ ]:
def reflection_loop(draft, critique_fn, revise_fn, max_rounds=3):
    for round_no in range(max_rounds):
        critique = critique_fn(draft)
        if critique['pass']:
            return draft
        draft = revise_fn(draft, critique)
    return draft

def critique_fn(text):
    return {'pass': len(text) < 100, 'reason': 'Too long'}

def revise_fn(text, critique):
    return text[:100]

print(reflection_loop('x' * 140, critique_fn, revise_fn))

## 5. Reflection can be expensive

Two additional model calls may improve quality but increase:
- latency
- token usage
- cost

Use reflection where the error cost justifies the extra work.

## 6. Evaluation and practice

Measure:
- baseline correctness
- post-reflection correctness
- false rejections
- extra cost
- extra latency

Practice: create a math answer verifier and an evidence-grounding verifier.

## Evaluation

Reflection is useful only if it improves the benchmark.

Improve with:
- deterministic checks first
- structured critique
- bounded retries
- specialized verifiers
- different model/role for critique when appropriate.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/